# Lab 6 - PID TempLABUdeA (Notebook Unificado)

Este notebook unifica:
- Cálculo de Kp, Ti, Td por ecuaciones en polo deseado (fsolve).
- Trazado del LGR.
- Ajuste PID por optimización para cumplir %OS, Ts y margen de fase.
- Simulación al escalón y guardado de resultados.

In [ ]:
# Si estas en Colab y faltan paquetes, descomenta esta celda:
# !pip -q install control scipy matplotlib numpy

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import control as ct
from pathlib import Path
from scipy.optimize import fsolve, differential_evolution

K = 0.293
TAU = 169.0
L_DELAY = 12.6

OUT_DIR = Path('Lab 6/figures')
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1) Cálculo por polo deseado (fsolve)

In [ ]:
sd = -0.0333 + 0.0455j

def sistema(vars, K, tau, sd):
    kp, ti, td = vars
    g_sd = K / (tau * sd + 1)
    gpid_sd = kp * (1 + 1/(ti * sd) + td * sd)
    l_sd = gpid_sd * g_sd
    err = l_sd + 1
    eq1 = np.real(err)
    eq2 = np.imag(err)
    eq3 = td - (ti / 4.0)
    return [eq1, eq2, eq3]

x0 = [100.0, TAU, TAU / 4.0]
sol = fsolve(lambda x: sistema(x, K, TAU, sd), x0)
kp_f, ti_f, td_f = sol

print(f'Kp = {kp_f:.6f}')
print(f'Ti = {ti_f:.6f} s')
print(f'Td = {td_f:.6f} s')

## 2) LGR + simulación al escalón con ajuste automático

In [ ]:
def pid_tf(kp, ti, td):
    s = ct.tf([1, 0], [1])
    return kp * (1 + 1/(ti*s) + td*s)

def plant_with_delay(k, tau, delay):
    g = ct.tf([k], [tau, 1])
    num_d, den_d = ct.pade(delay, 1)
    d = ct.tf(num_d, den_d)
    return g * d

def settling_time_2pct(t, y, yss):
    tol = 0.02 * abs(yss)
    idx = np.where(np.abs(y - yss) > tol)[0]
    if idx.size == 0: return 0.0
    return float(t[idx[-1] + 1]) if idx[-1] < len(t) - 1 else float(t[-1])

def eval_metrics(kp, ti, td):
    g = plant_with_delay(K, TAU, L_DELAY)
    c = pid_tf(kp, ti, td)
    l_open = c * g
    t_closed = ct.feedback(l_open, 1)
    
    t = np.linspace(0, 1200, 6000)
    t, y = ct.step_response(t_closed, t)
    yss = float(np.real(y[-1]))
    ymax = float(np.max(np.real(y)))
    os = max(0.0, (ymax - yss) / abs(yss) * 100.0)
    ts = settling_time_2pct(t, np.real(y), yss)
    
    gm, pm, wcg, wcp = ct.margin(l_open)
    pm = float(pm) if pm is not None else -180.0
    
    return {'os': float(os), 'ts': float(ts), 'pm': float(pm), 'yss': yss, 't': t, 'y': np.real(y), 'L': l_open, 'T': t_closed}

def objective(x):
    kp = 10 ** x[0]
    ti = 10 ** x[1]
    td = 10 ** x[2]
    m = eval_metrics(kp, ti, td)
    j_os = ((m['os'] - 10.0) / 10.0) ** 2
    j_ts = ((m['ts'] - 120.0) / 80.0) ** 2
    j_pm = ((max(0.0, 45.0 - m['pm'])) / 10.0) ** 2
    return j_os + j_ts + j_pm

# LGR
s = ct.tf([1, 0], [1])
lgr_open = K * (TAU * TAU/10 * s**2 + TAU * s + 1) / (TAU * s * (TAU * s + 1))

plt.figure(figsize=(8, 6))
ct.root_locus_plot(lgr_open, grid=True)
plt.title('Lugar Geometrico de las Raices')
plt.xlabel('Parte real')
plt.ylabel('Parte imaginaria')
plt.tight_layout()
plt.savefig(OUT_DIR / 'lgr_lab6.png', dpi=160)
plt.show()

# Optimizar PID
result = differential_evolution(objective, bounds=[(-2.0, 4.0), (0.0, 3.5), (-1.0, 3.0)], seed=7, maxiter=45, popsize=12)
kp = 10 ** result.x[0]
ti = 10 ** result.x[1]
td = 10 ** result.x[2]
m = eval_metrics(kp, ti, td)

print(f'Kp = {kp:.6f}')
print(f'Ti = {ti:.6f} s')
print(f'Td = {td:.6f} s')
print(f'%OS = {m["os"]:.4f}%')
print(f'Ts = {m["ts"]:.4f} s')
print(f'PM = {m["pm"]:.4f}°')

In [ ]:
# PID por interpolación (versión corta)
from scipy.interpolate import griddata

kp_grid = np.linspace(10, 120, 6)
ti_grid = np.linspace(20, 120, 6)
td_grid = np.linspace(1, 20, 6)

P, S = [], []
for kp_i in kp_grid:
    for ti_i in ti_grid:
        for td_i in td_grid:
            met = eval_metrics(kp_i, ti_i, td_i)
            if np.isfinite(met['os']) and np.isfinite(met['ts']) and met['pm'] >= 45:
                P.append([kp_i, ti_i, td_i])
                S.append([met['os'], met['ts']])

P = np.array(P, float)
S = np.array(S, float)
target = np.array([10.0, 120.0])

est = griddata(S, P, target, method='linear')
if est is None or np.any(np.isnan(est)):
    est = griddata(S, P, target, method='nearest')

kp_int, ti_int, td_int = np.asarray(est, float).reshape(-1)[:3]
m_int = eval_metrics(kp_int, ti_int, td_int)

print(f'Kp = {kp_int:.6f}')
print(f'Ti = {ti_int:.6f} s')
print(f'Td = {td_int:.6f} s')
print(f'%OS = {m_int["os"]:.4f}%')
print(f'Ts = {m_int["ts"]:.4f} s')
print(f'PM = {m_int["pm"]:.4f}°')

In [ ]:
# Gráfica simple de verificación con los parámetros interpolados
kp, ti, td = kp_int, ti_int, td_int
m = m_int

t = m['t']
y = m['y']
yss = m['yss']
os_limit = yss * 1.10

a = plt.figure(figsize=(8, 5))
plt.plot(t, y, 'b', linewidth=2, label='Respuesta simulada')
plt.axhline(os_limit, color='r', linestyle='--', linewidth=1.4, label='Límite de sobreimpulso (10%)')
plt.axvline(m['ts'], color='g', linestyle='--', linewidth=1.4, label=f"Ts = {m['ts']:.2f} s")
plt.title('Verificación de la respuesta al escalón')
plt.xlabel('Tiempo (s)')
plt.ylabel('Amplitud')
plt.grid(True, alpha=0.3)
plt.legend(loc='best')
plt.tight_layout()
plt.savefig(OUT_DIR / 'step_interpolated_lab6.png', dpi=160)
plt.show()

print('=== Resumen visual ===')
print(f'Kp = {kp:.6f}')
print(f'Ti = {ti:.6f} s')
print(f'Td = {td:.6f} s')
print(f'%OS = {m["os"]:.4f}%')
print(f'Ts = {m["ts"]:.4f} s')
print(f'PM = {m["pm"]:.4f}°')